## Problem: Add a Benchmark to Your PyTorch Code

### Problem Statement
Train a small fully connected classifier on MNIST and measure two things: how long each training epoch and each evaluation pass takes, and how fast the model's forward pass is on its own, using **`torch.utils.benchmark`**. Naive timing of PyTorch code, especially on a GPU, often gives wrong numbers, so the point of this exercise is to time things correctly.

### Requirements
1. **Model** (`SimpleNN`): flatten the 28×28 image, pass it through **at least two hidden `nn.Linear` layers with ReLU**, and output 10 logits.
2. **Epoch timing**: for every epoch, report the training time, the time to evaluate on the test set, the training throughput (images/s), and the test accuracy.
   - Use `time.perf_counter()`, which is monotonic and high-resolution, instead of `time.time()`.
   - CUDA kernels run **asynchronously**: a Python call returns as soon as the kernel is queued. Call `torch.cuda.synchronize()` before starting the clock and again before stopping it. Otherwise you only measure how long it took to launch the kernels.
   - Evaluate under `model.eval()` and `torch.inference_mode()`.
3. **Micro-benchmark** with `torch.utils.benchmark.Timer`: time the forward pass on a fixed batch with autograd recording and again under `torch.inference_mode()`, for two batch sizes. Print the results with `benchmark.Compare`. `Timer` runs warm-up iterations, synchronizes the GPU and repeats the measurement for you. Its `num_threads` defaults to **1**, so pass `torch.get_num_threads()` if you want numbers comparable to your training loop.

### Pitfalls an interviewer will probe
- **No warm-up**: the first iterations pay one-off costs such as CUDA context creation, memory-allocator growth, cuDNN autotuning and `torch.compile`. The first epoch is slower for this reason, so compare later epochs.
- **Missing synchronize** on a GPU, which makes the timings suspiciously fast.
- **Data loading dominates**: for a small MLP on MNIST, decoding and converting images in the `DataLoader` costs more than the model itself. Epoch time measures the whole pipeline. To measure the model alone, benchmark a batch that is already loaded, as in step 3.
- **Thread count**: CPU timings depend on `torch.get_num_threads()`, so only compare numbers measured with the same settings.

<details>
  <summary>💡 Hint</summary>
  Wrap each timed region as <code>sync(); start = time.perf_counter(); ...; sync(); elapsed = time.perf_counter() - start</code>, where <code>sync()</code> calls <code>torch.cuda.synchronize()</code> only when running on CUDA.
  <br>
  <code>benchmark.Timer(stmt="fn(model, x)", globals={...}, label=..., sub_label=..., description=...)</code> followed by <code>.blocked_autorange(min_run_time=0.5)</code> returns a <code>Measurement</code>. Collect them in a list and pass the list to <code>benchmark.Compare</code>.
</details>

In [ ]:
import time

import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.benchmark as benchmark
import torchvision
import torchvision.transforms as transforms

In [ ]:
class SimpleNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(28 * 28, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 10)

    def forward(self, x):
        x = x.flatten(1)  # (B, 1, 28, 28) -> (B, 784)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)  # logits: CrossEntropyLoss applies log-softmax itself

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def sync():
    """Block until all queued GPU work has finished, so the clock measures it. No-op on CPU."""
    if device.type == "cuda":
        torch.cuda.synchronize()

In [ ]:
# Load MNIST dataset
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])

train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)

test_dataset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64, shuffle=False)

In [ ]:
model = SimpleNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)


def evaluate(model, loader):
    model.eval()
    correct = total = 0
    with torch.inference_mode():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            correct += (model(images).argmax(dim=1) == labels).sum().item()
            total += labels.size(0)
    return correct / total


# Training loop with benchmarking. Epoch 1 includes one-off warm-up costs.
epochs = 5
for epoch in range(epochs):
    model.train()
    sync()
    start = time.perf_counter()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        loss = criterion(model(images), labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    sync()
    train_time = time.perf_counter() - start

    sync()
    start = time.perf_counter()
    accuracy = evaluate(model, test_loader)
    sync()
    test_time = time.perf_counter() - start

    print(f"Epoch [{epoch + 1}/{epochs}] loss {loss.item():.4f} | "
          f"train {train_time:.2f}s ({len(train_dataset) / train_time:,.0f} img/s) | "
          f"test {test_time:.2f}s, accuracy {100 * accuracy:.2f}%")

In [ ]:
# Micro-benchmark the forward pass alone, on batches that are already in memory
def forward(model, x):
    return model(x)


def forward_inference(model, x):
    with torch.inference_mode():  # no autograd graph, no saved activations
        return model(x)


model.eval()
results = []
for batch_size in (64, 1024):
    x = torch.randn(batch_size, 1, 28, 28, device=device)
    for name, fn in (("autograd", forward), ("inference_mode", forward_inference)):
        timer = benchmark.Timer(
            stmt="fn(model, x)",
            globals={"fn": fn, "model": model, "x": x},
            label="SimpleNN forward",
            sub_label=f"batch {batch_size}",
            description=name,
            num_threads=torch.get_num_threads(),  # Timer defaults to a single thread
        )
        results.append(timer.blocked_autorange(min_run_time=0.5))  # warms up, syncs, repeats

benchmark.Compare(results).print()